# 02 AAV LLM Agent

This notebook keeps the original workflow, with reusable logic moved to `llm_agent_core.py`. Run cells from top to bottom. Set `RUN_OPENAI` and `RUN_ESM` before executing expensive steps.

## Dependencies

Use the project virtual environment. If dependencies are missing, install: `openai python-dotenv fair-esm pydantic tqdm pandas numpy torch`.

In [ ]:
from pathlib import Path

import os

import esm
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from llm_agent_core import (
    AgentPaths,
    attach_candidate_truth,
    build_analyst_context,
    build_critic_candidates,
    build_designer_pool,
    candidate_records,
    compact_candidate_records,
    choose_device,
    evaluate_fitness,
    generate_hypotheses,
    load_fitness_model,
    load_two_vs_many,
    merge_designed_candidates,
    merge_final_recommendations,
    mutation_designer,
    mutation_frequency_by_position,
    run_data_analyst,
    scientific_critic,
    summarize_history,
)

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
RUN_OPENAI = False
RUN_ESM = False

LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
DESIGNER_POOL_SIZE = 40
DESIGNER_SELECTION_SIZE = 30
TOP_K = 10
BATCH_SIZE = 16

load_dotenv()
print("LLM model:", LLM_MODEL)
print("OpenAI key loaded:", os.getenv("OPENAI_API_KEY") is not None)

## Prepare Historical Evidence And Candidate Pool

In [ ]:
observed_df, candidate_pool = load_two_vs_many(AgentPaths.data)
top_history, mutation_summary = summarize_history(observed_df)
analyst_context = build_analyst_context(observed_df, top_history, mutation_summary)
designer_pool = build_designer_pool(
    candidate_pool,
    mutation_summary,
    pool_size=DESIGNER_POOL_SIZE,
)
records = compact_candidate_records(designer_pool, limit=DESIGNER_POOL_SIZE)

print("Observed experiments:", len(observed_df))
print("Unknown candidates:", len(candidate_pool))
print("Designer candidate pool:", len(designer_pool))
display(top_history[["mutated_region", "target"]].head(10))
display(mutation_summary.head(20))
display(designer_pool[["candidate_id", "mutated_region", "mutations", "num_mutations", "historical_prior"]].head(20))

## Mutation Landscape

This plot shows mutation frequency by amino-acid position among the top historical variants.


In [ ]:
landscape_df = mutation_frequency_by_position(top_history["mutated_region"])
landscape_df.to_csv(ARTIFACTS / "mutation_landscape_frequency.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(
    landscape_df["position"],
    landscape_df["mutation_frequency"],
    color="#2f6f73",
    width=0.8,
)
ax.set_xlabel("Amino-acid position")
ax.set_ylabel("Mutation frequency")
ax.set_title("Mutation landscape in top historical variants")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(ARTIFACTS / "mutation_landscape_frequency.png", dpi=200)
plt.show()

display(landscape_df.sort_values("mutation_frequency", ascending=False).head(10))


## Run LLM Analyst And Designer

In [ ]:
if RUN_OPENAI:
    client = OpenAI()
    analyst_report = run_data_analyst(client, LLM_MODEL, analyst_context)
    hypotheses = generate_hypotheses(client, LLM_MODEL, analyst_report)
    designer_result = mutation_designer(
        client,
        LLM_MODEL,
        analyst_report,
        hypotheses,
        records,
        n=DESIGNER_SELECTION_SIZE,
        max_candidates_for_llm=DESIGNER_POOL_SIZE,
        fallback_on_error=True,
    )
    designed_candidates = merge_designed_candidates(designer_pool, designer_result)
    test_truth = pd.read_csv("test.csv")
    designed_candidates = attach_candidate_truth(designed_candidates, test_truth)

    print(analyst_report)
    print(hypotheses)
    if any("local fallback" in item.reason for item in designer_result.selected):
        print("Mutation Designer used local fallback because the LLM API call failed or was rate-limited.")
    display(designed_candidates[[
        "candidate_id",
        "mutated_region",
        "mutations",
        "num_mutations",
        "historical_prior",
        "true_fitness",
        "reason",
    ]])
else:
    print("Set RUN_OPENAI = True to call the OpenAI API and create designed_candidates.")

## Score Designed Candidates With ESM-2 + Fitness Head

In [ ]:
if RUN_ESM:
    if "designed_candidates" not in globals():
        raise RuntimeError("Run the LLM designer cell first so designed_candidates exists.")

    device = choose_device()
    esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
    esm_model = esm_model.to(device).eval()
    for parameter in esm_model.parameters():
        parameter.requires_grad = False

    fitness_model = load_fitness_model(AgentPaths.checkpoint, device)
    scored_candidates = evaluate_fitness(
        designed_candidates,
        esm_model,
        alphabet.get_batch_converter(),
        fitness_model,
        device,
        batch_size=BATCH_SIZE,
    )

    display(scored_candidates[[
        "candidate_id",
        "mutations",
        "historical_prior",
        "predicted_fitness",
        "true_fitness",
        "prediction_error",
        "reason",
    ]].sort_values("predicted_fitness", ascending=False))

    top_k = (
        scored_candidates
        .sort_values("predicted_fitness", ascending=False)
        .head(TOP_K)
        .copy()
        .reset_index(drop=True)
    )
    top_k.insert(0, "rank", range(1, len(top_k) + 1))
    display(top_k[[
        "rank",
        "candidate_id",
        "mutated_region",
        "mutations",
        "predicted_fitness",
        "true_fitness",
        "prediction_error",
        "reason",
    ]])
else:
    print("Set RUN_ESM = True after designed_candidates exists to score candidates.")

## Scientific Critic And Final Recommendations

In [ ]:
if RUN_OPENAI and "top_k" in globals():
    critic_result = scientific_critic(
        client,
        LLM_MODEL,
        build_critic_candidates(top_k),
        analyst_report,
        hypotheses,
        fallback_on_error=True,
    )
    if "fallback" in critic_result.overall_limitations.lower():
        print("Scientific Critic used local fallback because the LLM API call failed or was rate-limited.")
    final_recommendations = merge_final_recommendations(top_k, critic_result)
    test_truth = pd.read_csv("test.csv")
    final_recommendations = attach_candidate_truth(final_recommendations, test_truth)
    final_recommendations.to_csv("artifacts/llm_agent_final_recommendations.csv", index=False)

    display(final_recommendations[[
        "rank",
        "candidate_id",
        "mutations",
        "predicted_fitness",
        "true_fitness",
        "prediction_error",
        "priority",
        "recommendation_reason",
        "supporting_evidence",
        "uncertainty",
    ]])

    print("========== OVERALL RECOMMENDATION ==========")
    print(critic_result.overall_recommendation)
    print("\n========== MAJOR LIMITATIONS ==========")
    print(critic_result.overall_limitations)
else:
    print("Run the OpenAI and ESM sections first to generate final recommendations.")